In [1]:
import sys

sys.path.insert(0, "/Users/shelleygoel/Code/01_statistical_mod_blog/anomaly_detection")

import re
from datetime import datetime
from pathlib import Path

import numpy as np

import pandas as pd
import plotly.express as px
from core.dataset import TimeSeriesDataset
from core.evaluation import Evaluation
from core.feature_transformer import C22Feature, FeatureCategory, FeatureTransformer
from core.hvac_data_gen import HVACDataGenerator
from core.models import Catch22MPModel, EuclideanDistModel, FeatureWeighter, IForestModel, UniformWeighter
from core.viz import plot_cases
from sklearn.tree import DecisionTreeClassifier

/Users/shelleygoel/miniconda3/envs/TSB-AD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load HVAC Data

In [2]:
skip_data_gen = True
if not skip_data_gen:
    # Generate HVAC Data with longer history
    generator = HVACDataGenerator(seed=10)
    hvac_df = generator.generate_dataset(
        num_containers=500,
        start_time=datetime(2026, 1, 15),
        duration_days=20,
    )
else:
    hvac_df = pd.read_parquet(Path("../datasets/hvac_anomalies_v021226.parquet"))

    hvac_df = hvac_df.copy()

# Post processing: smooth TmpRet    
hvac_df["TmpRet"] = hvac_df.groupby(["container_id", "unit"])["TmpRet"].transform(
        lambda x: x.rolling(window=10, min_periods=1).mean()
    )

# Wrap in TimeSeriesDataset
col_map = {
    "entity": "container_id",
    "time": "timestamp_et",
    "value_cols": ["TmpRet"],
    "label": "anomaly",
    "label_type": "anomaly_type",
    "sub_entity": "unit",
}
hvac_ds = TimeSeriesDataset(hvac_df, col_map)
print(hvac_ds.anomaly_summary())

sample_for_exp = False
if sample_for_exp:
    n_sample_size = 100
    sampled_entities = hvac_ds.sample_entities(n_cases=50, label_type="frequency", random_state=42)
    normal_entities = hvac_ds.sample_entities(n_cases=n_sample_size, label_type="normal", random_state=42)

    train_entities = np.concatenate([sampled_entities, normal_entities])
    hvac_ds = TimeSeriesDataset(hvac_df[hvac_df["container_id"].isin(train_entities)], col_map)
    print(hvac_ds.anomaly_summary())

  label_type  entity_count
0     normal           911
1        lag            38
2  amplitude            27
3  frequency            24



# Euclidean Distance Model


In [3]:

eucl = EuclideanDistModel(feature_col="TmpRet", smooth_window=1, dist_window=12 * 60, strategy="iqr")
eucl_day_scores = eucl.score_anomalies(hvac_ds, level="day")

# Catch22 MP Model

## C22 Features Calculation

In [4]:
# 3. Feature transform
feats_to_calc = [
    C22Feature.CO_f1ecac,
    C22Feature.CO_FirstMin_ac,
    C22Feature.IN_AutoMutualInfoStats_40_gaussian_fmmi,
    C22Feature.SP_Summaries_welch_rect_area_5_1,
    C22Feature.SP_Summaries_welch_rect_centroid,
]
ft = FeatureTransformer(
    raw_data_columns=["TmpRet"], window_size=12 * 60, stride=60, n_jobs=8, c22_features=feats_to_calc
)
feat_ds = ft.transform(hvac_ds)  # categories=[FeatureCategory.C22_RAW_DIFF])

FeatureTransformer: 100%|██████████| 1000/1000 [00:53<00:00, 18.78it/s]


## C22MP: Score Anomalies
- Left C22MP Profile Calculation
- day Level Scores

In [5]:
calc_features = sum([list(v) for v in ft.feature_map.values()], [])

class CustomWeighter(FeatureWeighter):
    def compute_weights(self, preferred_features: list) -> dict:
        return dict([(feat, 1) for feat in preferred_features])


# pattern = re.compile(
#     r"(?=.*_featdiff)(?=.*(SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid|PD_PeriodicityWang_th0.01|CO_f1ecac|CO_FirstMin_ac|IN_AutoMutualInfoStats_40_gaussian_fmmi))"
# )
# pattern = re.compile(
#     r"(?=.*_featdiff)(?=.*(SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid|CO_f1ecac))" 
# )
# pattern = re.compile(
#     r"^(?!.*_featdiff).*(SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid|CO_f1ecac)"
# )
# pattern = re.compile(r'(?=.*_featdiff)(?=.*(CO_f1ecac))')
# pattern = re.compile(r"^(?!.*_featdiff).*(0_1__CO_f1ecac|0_2__CO_f1ecac|1_2__CO_f1ecac)")
# pattern = re.compile(r'SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid')

cw = CustomWeighter()
pref_feat_exp = [
    ("C22 Raw Feat", ["TmpRet__0__CO_f1ecac", "TmpRet__1__CO_f1ecac", "TmpRet__2__CO_f1ecac"]),
    ("C22 Feat Diff",
        [
            "TmpRet__featdiff_0_1__CO_f1ecac",
            "TmpRet__featdiff_0_2__CO_f1ecac",
            "TmpRet__featdiff_1_2__CO_f1ecac",
        ],
    ),
    ("C22 Raw Diff Feat", ["TmpRet__0_1__CO_f1ecac", "TmpRet__0_2__CO_f1ecac", "TmpRet__1_2__CO_f1ecac"]),
]
stride = ft.stride  # 45
exclude_zone = 1440 // stride  # 32
print(f"{stride=}, {exclude_zone=}")
c22_mps = {}
c22_day_scores = {}
for model_name, pref_features in pref_feat_exp:
    custom_wei = cw.compute_weights(preferred_features=pref_features)
    # 4. Fit MP once
    c22mp = Catch22MPModel(
        exclude_zone=exclude_zone,
        early_abandon=False,  # brute force — fast enough
    )
    mp_ds, debug = c22mp.fit_profile(feat_ds, weights=custom_wei)
    scores = c22mp.score(mp_ds, level="day", day_agg_stat="p90")
    c22_mps[model_name] = mp_ds
    c22_day_scores[model_name] = scores


stride=60, exclude_zone=24


Catch22MP fit_profile: 100%|██████████| 1000/1000 [00:00<00:00, 1684.77it/s]


In [6]:
scores_long = pd.concat(
    [s.df[["anomaly_score"]].dropna().assign(model=name) for name, s in c22_day_scores.items()],
    ignore_index=True,
)
px.histogram(scores_long, x="anomaly_score", color="model", barmode="overlay", opacity=0.5)



**Observations **
- the score distributions are left skewed as expected so this is a good sanity check

# Isolation Forest Model

In [8]:
iforest_day_scores = {} 
hand_selected_feats = [
            "TmpRet__featdiff_0_1__CO_f1ecac",
            "TmpRet__featdiff_0_2__CO_f1ecac",
            "TmpRet__featdiff_1_2__CO_f1ecac",
        ]

iforest = IForestModel(fit_scope="global", contamination=0.02, feature_cols=hand_selected_feats)
iforest_day_scores["IForest Hand Select Feat"] = iforest.score_anomalies(feat_ds, level="day", day_agg_stat="p90")
px.histogram(iforest_day_scores["IForest Hand Select Feat"].df["anomaly_score"])

# Compare Models: PR curves

In [9]:
ev = Evaluation(level="day")
fig = ev.plot_pr_curves_compared(
    # {"Catch22MP": c22_day_scores},
    {**c22_day_scores, "eucl_dist": eucl_day_scores, **iforest_day_scores},
    hvac_ds,
)
fig.show()

# Feature Selection using Classification
- using a sampled of labeled data
- timestamp labels aggregated to window level label
  - Window is labeled as anomaly if even a single timestamp is anomalous - since we want to have high recall.
- fit a decision tree classifier - using balanced accuracy as measure as anomalies are rate
- to find features which can separate the normal from anomalous cases
- Selected Features can be used in any model - MP or Iforest

In [13]:
sampled_entities = hvac_ds.sample_entities(n_cases=10, label_type="frequency", random_state=42)
normal_entities = hvac_ds.sample_entities(n_cases=100, label_type="normal", random_state=42)

train_entities = np.concatenate([sampled_entities, normal_entities])

In [14]:

entity_col = feat_ds.col_map["entity"]

feat_ds_train = TimeSeriesDataset(
    feat_ds.df[feat_ds.df[entity_col].isin(train_entities)].copy(),
    feat_ds.col_map,
)
ds_train = TimeSeriesDataset(hvac_df[hvac_df["container_id"].isin(train_entities)], col_map)

In [15]:
ds_train.anomaly_summary()

,label_type,entity_count
0,normal,100
1,frequency,10


In [29]:

window_size = ft.window_size
stride = ft.stride
entity_col = hvac_ds.col_map["entity"]
time_col = hvac_ds.col_map["time"]
label_col = hvac_ds.col_map["label"]

ts_labels_df = ds_train.ts_labels()  # entity, time, label, [label_type]

# --- 1. Label each feature window ---
# Treat it as a "feature on the label": max(label) over [t, t+window_size),
# strided by ft.stride. Vectorized via pivot to (time, entity) then numpy
# strided indexing — mirrors FeatureTransformer's windowing.
label_mat = (
    ts_labels_df.pivot(index=time_col, columns=entity_col, values=label_col)
    .sort_index().astype(int)
)
n_windows = (len(label_mat) - window_size) // stride + 1
starts = np.arange(n_windows) * stride
window_idx = starts[:, None] + np.arange(window_size)[None, :]  # (n_windows, W)

window_label_mat = label_mat.values[window_idx].max(axis=1)  # (n_windows, n_entities)
window_labels_df = (
    pd.DataFrame(window_label_mat, index=label_mat.index.values[starts], columns=label_mat.columns)
    .rename_axis(time_col).reset_index()
    .melt(id_vars=time_col, var_name=entity_col, value_name="window_label")
)

# --- 2. Build (X, y) ---
merged = feat_ds_train.df.merge(window_labels_df, on=[entity_col, time_col])

feature_cols = feat_ds_train.col_map["value_cols"]
# for frequency anomaly we know that 
feature_cols = ft.feature_map[FeatureCategory.C22_FEAT_DIFF]
X = merged[feature_cols].values

y = merged["window_label"].astype(int).values

import numpy as np                                                                                                                                                                                                                                                                                                
from scipy.stats import rankdata
def per_feature_auc_roc(X: np.ndarray, y: np.ndarray) -> np.ndarray:                                                                                                                                                                                                                                              
    """AUC-ROC per feature; NaN-safe via per-column masking."""
    y = np.asarray(y).astype(bool)                                                                                                                                                                                                                                                                                
    n_features = X.shape[1]                                                                                                                                                                                                                                                                                       
    out = np.full(n_features, np.nan)
                                                                                                                                                                                                                                                                                                                
    for j in range(n_features):                                                                                                                                                                                                                                                                                   
        col = X[:, j]
        mask = ~np.isnan(col)            # drop NaN rows for THIS feature                                                                                                                                                                                                                                         
        yj = y[mask]                                      
        n_pos = yj.sum()                                                                                                                                                                                                                                                                                          
        n_neg = (~yj).sum()
        if n_pos == 0 or n_neg == 0:                                                                                                                                                                                                                                                                              
            continue                      # one class gone after masking — skip                                                                                                                                                                                                                                   
                                                                                                                                                                                                                                                                                                                
        ranks = rankdata(col[mask])                                                                                                                                                                                                                                                                               
        U = ranks[yj].sum() - n_pos * (n_pos + 1) / 2                                                                                                                                                                                                                                                             
        out[j] = U / (n_pos * n_neg)                                                                                                                                                                                                                                                                              

    return out

#Direction-agnostic "separability" — useful when you don't care
  # whether high or low values indicate anomaly.
def per_feature_separability(X, y):
    auc = per_feature_auc_roc(X, y)
    return np.maximum(auc, 1 - auc)          # in [0.5, 1.0]

scores = per_feature_separability(X, y)           # (130,)
top = np.argsort(scores)[::-1][:10]               # top 10 features
roc_selected_feat = np.array(feature_cols)[top]
print(roc_selected_feat)                     


['TmpRet__featdiff_0_1__CO_FirstMin_ac' 'TmpRet__featdiff_0_1__CO_f1ecac'
 'TmpRet__featdiff_0_1__SP_Summaries_welch_rect_centroid'
 'TmpRet__featdiff_0_2__CO_FirstMin_ac' 'TmpRet__featdiff_0_2__CO_f1ecac'
 'TmpRet__featdiff_0_2__SP_Summaries_welch_rect_centroid'
 'TmpRet__featdiff_0_2__SP_Summaries_welch_rect_area_5_1'
 'TmpRet__featdiff_1_2__CO_FirstMin_ac'
 'TmpRet__featdiff_0_1__SP_Summaries_welch_rect_area_5_1'
 'TmpRet__featdiff_0_2__IN_AutoMutualInfoStats_40_gaussian_fmmi']


In [22]:


print(f"n_windows={len(y)}, n_anomaly_windows={y.sum()}, base_rate={y.mean():.3f}")

# --- 3. Fit Decision Tree + rank features ---
clf = DecisionTreeClassifier(
    max_depth=10,
    class_weight="balanced",  # anomalies are rare — rebalance
    random_state=42,
)
clf.fit(X, y)

feat_imp = (
    pd.DataFrame({"feature": feature_cols, "importance": clf.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
dt_selected_feat = feat_imp.head(10)["feature"].values
dt_selected_feat

n_windows=17270, n_anomaly_windows=576, base_rate=0.033


array(['TmpRet__0_2__CO_f1ecac', 'TmpRet__0_1__CO_f1ecac',
       'TmpRet__0_2__SP_Summaries_welch_rect_area_5_1',
       'TmpRet__featdiff_0_2__SP_Summaries_welch_rect_area_5_1',
       'TmpRet__0_1__SP_Summaries_welch_rect_area_5_1',
       'TmpRet__0__SP_Summaries_welch_rect_area_5_1',
       'TmpRet__1_2__CO_f1ecac',
       'TmpRet__0_2__IN_AutoMutualInfoStats_40_gaussian_fmmi',
       'TmpRet__featdiff_0_2__CO_f1ecac',
       'TmpRet__1__SP_Summaries_welch_rect_area_5_1'], dtype=object)

In [32]:
exp = [
    ("IForest ROC Feat(5)", roc_selected_feat[:5]),
    ("IForest ROC Feat(3)", roc_selected_feat[:3]),
    ("IForest DT Feat", dt_selected_feat[:3])

]
for md, feat in exp:
    iforest = IForestModel(fit_scope="global", contamination=0.02, feature_cols=feat)
    iforest_day_scores[md] = iforest.score_anomalies(feat_ds, level="day", day_agg_stat="p90")
    px.histogram(iforest_day_scores[md].df["anomaly_score"]).show()


In [33]:
ev = Evaluation(level="day")
fig = ev.plot_pr_curves_compared(
    # {k: c22_day_scores[k] for k in ["feat_diff", "clf_feat_select"]} | {"iforest_clf_select": iforest_day_scores_v2, "iforest_hand_select": iforest_day_scores_v1, "iforest_v3": iforest_day_scores_v3},
    iforest_day_scores,
    hvac_ds,
)
fig.show()

## Sample cases at boundary of score threshold

In [ ]:
threshold = 0.7
false

# Appendix

### Features for frequency
Direct spectral (Welch power spectrum)
  - SP_Summaries_welch_rect_area_5_1 — power in the lowest 1/5 of the spectrum. Shifts when dominant frequency moves.
  - SP_Summaries_welch_rect_centroid — frequency at which power is concentrated. Clearest signal for "wrong period."

  Periodicity detectors
  - PD_PeriodicityWang_th0.01 — Wang's dominant-periodicity estimate. Directly encodes the cycle length.
  - CO_f1ecac — first crossing of ACF at 1/e. A characteristic timescale — changes with period.
  - CO_FirstMin_ac — first minimum of the autocorrelation function. Roughly half the dominant period.
  - IN_AutoMutualInfoStats_40_gaussian_fmmi — first minimum of Auto-Mutual-Information. Nonlinear analog of CO_FirstMin_ac.

  Scaling / long-range (weaker, indirect)
  - SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1 — DFA scaling exponent.
  - SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1 — R/S range fit. Both reflect how power distributes across scales.

  Residual-autocorrelation
  - FC_LocalSimple_mean1_tauresrat — ratio of residual ACF timescale to raw ACF timescale after a simple forecast. Sensitive to how "learnable" the periodic structure is.

In [ ]:
feat_cfg = feat_ds.to_plot_cfg()
feat_cfg.value_cols = (
    # pref_features  # sorted([col for col in feat_ds.to_plot_cfg().value_cols if col.find("0_2") != -1] )
    pref_feat_exp[1][1]
)
skip = False
lbt = "frequency"
# lbt = "normal"
if not skip:
    figs = plot_cases(
        # [ds_small.to_plot_cfg(),
        [hvac_ds.to_plot_cfg(), mp_ds.to_plot_cfg(), feat_cfg],
        sample_from=hvac_ds,
        n_cases=10,
        # entity_ids=[246],
        # label_type="frequency",
        label_type=lbt,
        labels_from=hvac_ds.to_plot_cfg(),
        random_state=42,
    )


ckpt_dir = Path(f"checkpoints/hvac/{lbt}")
ckpt_dir.mkdir(exist_ok=True)

for i, fig in enumerate(figs):
    fig.write_html(ckpt_dir / f"case_{i}.html")
    # fig.write_image(ckpt_dir / f"case_{i}.png", width=1200, height=600, scale=2)
    # fig.show()

# 7. Evaluate
# ev = Evaluation(level='day')
# print(ev.metrics_table(day_scores, ds_small))